In [14]:
# Imports
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import math
import random
from collections import Counter

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
# Paths
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/Datasets/CS584"
CSV_PATH = f"{BASE_DIR}/movies.csv"  # make sure this file exists
OUTPUT_DIR = f"{BASE_DIR}/Outputs/q1"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("CSV_PATH:", CSV_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

CSV_PATH: /content/drive/MyDrive/Colab Notebooks/Datasets/CS584/movies.csv
OUTPUT_DIR: /content/drive/MyDrive/Colab Notebooks/Datasets/CS584/Outputs/q1


In [ ]:
# -------------------------
# Config 
# -------------------------
MAX_VOCAB_SIZE = 20000
MIN_FREQ = 2
MAX_LEN = 128

BATCH_SIZE = 64
EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.1

NUM_EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 1e-2

RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


Using device: cuda


In [ ]:

# -------------------------
# Tokenization / Vocab
# -------------------------

def simple_tokenize(text):
    text = text.lower()
    return re.findall(r"\b\w+\b", text)

def build_vocab(texts, max_size=MAX_VOCAB_SIZE, min_freq=MIN_FREQ):
    counter = Counter()
    for t in texts:
        counter.update(simple_tokenize(t))

    PAD = "<pad>"
    UNK = "<unk>"
    stoi = {PAD: 0, UNK: 1}
    idx = 2

    for word, freq in counter.most_common():
        if freq < min_freq:
            continue
        if idx >= max_size:
            break
        stoi[word] = idx
        idx += 1

    itos = {i: s for s, i in stoi.items()}
    return stoi, itos

def encode_text(text, stoi, max_len=MAX_LEN):
    tokens = simple_tokenize(text)
    ids = [stoi.get(tok, stoi["<unk>"]) for tok in tokens]
    ids = ids[:max_len]
    if len(ids) < max_len:
        ids = ids + [stoi["<pad>"]] * (max_len - len(ids))
    return ids

# -------------------------
# Dataset
# -------------------------

class ReviewsDataset(Dataset):
    def __init__(self, texts, labels, stoi, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.stoi = stoi
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = encode_text(self.texts[idx], self.stoi, self.max_len)
        x = torch.tensor(ids, dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

# -------------------------
# Positional Encoding
# -------------------------

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len]
        return self.dropout(x)

# -------------------------
# Simple Transformer Encoder Block
# -------------------------

class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim, dropout):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads, batch_first=True, dropout=dropout
        )
        self.linear1 = nn.Linear(d_model, ff_dim)
        self.linear2 = nn.Linear(ff_dim, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.act = nn.ReLU()

    def forward(self, x, src_key_padding_mask=None):
        # Self-attention
        attn_out, _ = self.self_attn(
            x, x, x, key_padding_mask=src_key_padding_mask
        )
        x = x + self.dropout1(attn_out)
        x = self.norm1(x)

        # Feed-forward
        ff = self.linear2(self.dropout(self.act(self.linear1(x))))
        x = x + self.dropout2(ff)
        x = self.norm2(x)
        return x

# -------------------------
# Transformer Classifier
# -------------------------

class TransformerSentimentClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=EMBED_DIM,
        n_heads=NUM_HEADS,
        num_layers=NUM_LAYERS,
        ff_dim=FF_DIM,
        max_len=MAX_LEN,
        num_classes=2,
        dropout=DROPOUT,
        pad_idx=0,
    ):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_enc = PositionalEncoding(d_model, max_len=max_len, dropout=dropout)

        self.layers = nn.ModuleList(
            [EncoderBlock(d_model, n_heads, ff_dim, dropout) for _ in range(num_layers)]
        )

        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, input_ids):
        # input_ids: (batch, seq_len)
        mask = (input_ids == self.pad_idx)     # (batch, seq_len)
        x = self.embedding(input_ids)          # (batch, seq_len, d_model)
        x = self.pos_enc(x)

        for layer in self.layers:
            x = layer(x, src_key_padding_mask=mask)

        x = self.norm(x)

        # mean pooling over non-pad tokens
        not_pad = (~mask).unsqueeze(-1)       # (batch, seq_len, 1)
        x = x * not_pad
        lengths = not_pad.sum(dim=1).clamp(min=1)
        pooled = x.sum(dim=1) / lengths       # (batch, d_model)

        pooled = self.dropout(pooled)
        logits = self.fc(pooled)              # (batch, num_classes)
        return logits

# -------------------------
# Training helpers
# -------------------------

def accuracy_from_logits(logits, labels):
    preds = logits.argmax(dim=1)
    return (preds == labels).float().mean().item()

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    total_count = 0

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        bs = y.size(0)
        total_loss += loss.item() * bs
        total_acc += accuracy_from_logits(logits, y) * bs
        total_count += bs

    return total_loss / total_count, total_acc / total_count

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    total_count = 0

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)
        loss = criterion(logits, y)

        bs = y.size(0)
        total_loss += loss.item() * bs
        total_acc += accuracy_from_logits(logits, y) * bs
        total_count += bs

    return total_loss / total_count, total_acc / total_count

# -------------------------
# Main
# -------------------------

def main():
    # Load data
    df = pd.read_csv(CSV_PATH)
    texts = df["review"].astype(str).tolist()

    # Map sentiment strings to 0/1
    label_map = {"negative": 0, "positive": 1}
    labels = [label_map[s.strip().lower()] for s in df["sentiment"]]

    # Build vocab
    stoi, itos = build_vocab(texts)
    vocab_size = len(stoi)
    print("Vocab size:", vocab_size)

    # Train/val/test split
    X_temp, X_test, y_temp, y_test = train_test_split(
        texts, labels, test_size=0.1, random_state=RANDOM_SEED, stratify=labels
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.1, random_state=RANDOM_SEED, stratify=y_temp
    )

    print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

    # Datasets and loaders
    train_ds = ReviewsDataset(X_train, y_train, stoi, max_len=MAX_LEN)
    val_ds   = ReviewsDataset(X_val, y_val, stoi, max_len=MAX_LEN)
    test_ds  = ReviewsDataset(X_test, y_test, stoi, max_len=MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    # Model, loss, optimizer
    model = TransformerSentimentClassifier(
        vocab_size=vocab_size,
        d_model=EMBED_DIM,
        n_heads=NUM_HEADS,
        num_layers=NUM_LAYERS,
        ff_dim=FF_DIM,
        max_len=MAX_LEN,
        num_classes=2,
        dropout=DROPOUT,
        pad_idx=stoi["<pad>"],
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    # Training loop
    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        print(
            f"Epoch {epoch:03d}/{NUM_EPOCHS} | "
            f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}"
        )

    # Final test evaluation
    test_loss, test_acc = evaluate(model, test_loader, criterion)
    print(f"\nFinal Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

    # Save model
    model_path = os.path.join(OUTPUT_DIR, "model.pt")
    torch.save(model.state_dict(), model_path)
    print("Saved model to:", model_path)

    # Save metrics
    metrics_path = os.path.join(OUTPUT_DIR, "metrics.txt")
    with open(metrics_path, "w") as f:
        for epoch in range(len(train_losses)):
            f.write(
                f"Epoch {epoch+1}: "
                f"train_loss={train_losses[epoch]:.4f}, "
                f"val_loss={val_losses[epoch]:.4f}, "
                f"train_acc={train_accs[epoch]:.4f}, "
                f"val_acc={val_accs[epoch]:.4f}\n"
            )
        f.write(f"\nFinal test_loss={test_loss:.4f}, test_acc={test_acc:.4f}\n")
    print("Saved metrics to:", metrics_path)

    # Plots
    epochs_range = range(1, len(train_losses) + 1)

    plt.figure()
    plt.plot(epochs_range, train_losses, label="Train Loss")
    plt.plot(epochs_range, val_losses, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss vs Epoch")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    loss_plot_path = os.path.join(OUTPUT_DIR, "loss.png")
    plt.savefig(loss_plot_path)
    plt.close()
    print("Saved loss plot to:", loss_plot_path)

    plt.figure()
    plt.plot(epochs_range, train_accs, label="Train Acc")
    plt.plot(epochs_range, val_accs, label="Val Acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Accuracy vs Epoch")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    acc_plot_path = os.path.join(OUTPUT_DIR, "accuracy.png")
    plt.savefig(acc_plot_path)
    plt.close()
    print("Saved accuracy plot to:", acc_plot_path)

    print("\nPushed to Drive, ", OUTPUT_DIR)

# -------------------------
# Run
# -------------------------
main()


Vocab size: 20000
Train: 40500, Val: 4500, Test: 5000
Epoch 001/100 | Train Loss: 0.5363, Acc: 0.7169 | Val Loss: 0.4380, Acc: 0.7958
Epoch 002/100 | Train Loss: 0.3978, Acc: 0.8193 | Val Loss: 0.3847, Acc: 0.8278
Epoch 003/100 | Train Loss: 0.3415, Acc: 0.8511 | Val Loss: 0.3559, Acc: 0.8416
Epoch 004/100 | Train Loss: 0.3008, Acc: 0.8725 | Val Loss: 0.3534, Acc: 0.8522
Epoch 005/100 | Train Loss: 0.2709, Acc: 0.8873 | Val Loss: 0.3593, Acc: 0.8553
Epoch 006/100 | Train Loss: 0.2438, Acc: 0.9031 | Val Loss: 0.3773, Acc: 0.8484
Epoch 007/100 | Train Loss: 0.2180, Acc: 0.9138 | Val Loss: 0.3763, Acc: 0.8542
Epoch 008/100 | Train Loss: 0.1986, Acc: 0.9219 | Val Loss: 0.3888, Acc: 0.8540
Epoch 009/100 | Train Loss: 0.1817, Acc: 0.9294 | Val Loss: 0.4330, Acc: 0.8460
Epoch 010/100 | Train Loss: 0.1620, Acc: 0.9379 | Val Loss: 0.4769, Acc: 0.8518
Epoch 011/100 | Train Loss: 0.1497, Acc: 0.9418 | Val Loss: 0.4848, Acc: 0.8504
Epoch 012/100 | Train Loss: 0.1347, Acc: 0.9472 | Val Loss: 0.4851